# Tanzania — Climate Data Profiling, Cleaning, and EDA

Analyze Tanzania climate data for consistent profiling, cleaning, and basic exploratory analysis ahead of COP32.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


## 1) Data Loading and Date Parsing
Load the CSV, add a country label, and build a proper `DATE` column from `YEAR` and `DOY`.

In [ ]:
df = pd.read_csv("tanzania.csv")
df['Country'] = 'Tanzania'

# Replace NASA sentinel -999 values with NaN before profiling
df.replace(-999, np.nan, inplace=True)

# Convert YEAR and DOY into a datetime column and extract month
df["DATE"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["MONTH"] = df["DATE"].dt.month

print("Shape:", df.shape)
print("Date range:", df["DATE"].min().date(), "to", df["DATE"].max().date())
df.head()


## 2) Data Profiling
Profile the dataset and compute missing-value percentages for each column.

In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary[missing_summary["missing_count"] > 0]


### Missing-Value Report
List columns with greater than 5% nulls and interpret the impact on analysis.

## 3) Duplicate Detection
Identify and remove exact duplicate rows before cleaning.

In [ ]:
duplicate_mask = df.duplicated(keep=False)
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)
if duplicates:
    print("Columns involved in exact duplicate rows:", list(df.columns))
    display(df[duplicate_mask].head(5))
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)


### Duplicate Detection Result
Exact duplicates are removed to avoid bias in climate summary statistics.

## 4) Outlier Detection & Basic Cleaning
Compute Z-scores for key weather variables and handle missing values.

In [ ]:
clean_df = df.copy()
weather_cols = [
    "T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR",
    "RH2M", "WS2M", "WS2M_MAX"
]
for col in weather_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df = clean_df.sort_values("DATE").reset_index(drop=True)
z_scores = clean_df[weather_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
outlier_mask = z_scores.abs() > 3
outlier_count = outlier_mask.any(axis=1).sum()
print("Rows flagged as outliers (|Z|>3):", outlier_count)
print("Outlier counts by variable:")
print(outlier_mask.sum())

clean_df["outlier_flag"] = outlier_mask.any(axis=1)

clean_df[weather_cols] = clean_df[weather_cols].ffill()
row_missing_pct = clean_df.isna().mean(axis=1)
rows_to_drop = (row_missing_pct > 0.3).sum()
print("Rows with >30% missing values to drop:", rows_to_drop)
clean_df = clean_df[row_missing_pct <= 0.3].reset_index(drop=True)

os.makedirs("data", exist_ok=True)
clean_df.to_csv(r"data/tanzania_clean.csv", index=False)
print("Exported cleaned DataFrame to data/tanzania_clean.csv")


### Outlier and Missing-Value Cleaning Decisions
- Outliers are flagged with |Z| > 3 for key climate variables.
- Retaining outliers preserves real extreme weather events instead of censoring climate signals.
- Forward-fill is used to keep short gaps in weather series intact.
- Rows with more than 30% missing values are dropped to avoid unreliable imputation.
- Cleaned output is saved to `data/tanzania_clean.csv`.
